# Notebook 02 — Satisfação e Engajamento

**Pergunta central:** O que impacta a satisfação dos colaboradores?

Análise das relações entre satisfação, produtividade, feedback, salário, cargo e departamento.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('../data/raw/hr_dashboard_data.csv')
df['Joining Date'] = pd.to_datetime(df['Joining Date'], format='%b-%y')
df['anos_empresa'] = ((pd.Timestamp('2024-01-01') - df['Joining Date']).dt.days / 365).round(1)

# categorias de satisfação
df['nivel_satisfacao'] = pd.cut(
    df['Satisfaction Rate (%)'],
    bins=[0, 33, 66, 100],
    labels=['Baixa (0-33%)', 'Média (34-66%)', 'Alta (67-100%)']
)

print(f'Satisfação média: {df["Satisfaction Rate (%)"].mean():.1f}%')
print(f'Colaboradores com satisfação baixa (< 33%): {(df["Satisfaction Rate (%)"] < 33).sum()}')
df.head(3)

Satisfação média: 49.9%
Colaboradores com satisfação baixa (< 33%): 63


,Name,Age,Gender,Projects Completed,Productivity (%),Satisfaction Rate (%),Feedback Score,Department,Position,Joining Date,Salary,anos_empresa,nivel_satisfacao
0,Douglas Lindsey,25,Male,11,57,25,4.7,Marketing,Analyst,2020-01-01,63596,4.0,Baixa (0-33%)
1,Anthony Roberson,59,Female,19,55,76,2.8,IT,Manager,1999-01-01,112540,25.0,Alta (67-100%)
2,Thomas Miller,30,Male,8,87,10,2.4,IT,Analyst,2017-01-01,66292,7.0,Baixa (0-33%)


## 1. Satisfação por Departamento

In [3]:
sat_depto = df.groupby('Department')['Satisfaction Rate (%)'].mean().reset_index()
sat_depto.columns = ['departamento', 'satisfacao_media']
sat_depto = sat_depto.sort_values('satisfacao_media')

sat_depto['cor'] = sat_depto['satisfacao_media'].apply(
    lambda x: '#ef5350' if x < 45 else ('#ffa726' if x < 55 else '#66bb6a')
)

fig = px.bar(
    sat_depto,
    x='satisfacao_media', y='departamento',
    orientation='h',
    title='Satisfação Média por Departamento',
    labels={'satisfacao_media': 'Satisfação Média (%)', 'departamento': ''},
    color='satisfacao_media',
    color_continuous_scale=['#ef5350', '#ffa726', '#66bb6a'],
    text=sat_depto['satisfacao_media'].round(1).astype(str) + '%'
)
fig.update_traces(textposition='outside')
fig.update_layout(coloraxis_showscale=False)
fig.add_vline(x=50, line_dash='dash', line_color='gray', annotation_text='Média geral')
fig.show()

## 2. Satisfação por Cargo

In [4]:
ordem_cargos = ['Intern', 'Junior Developer', 'Analyst', 'Team Lead', 'Senior Developer', 'Manager']

sat_cargo = df.groupby('Position')['Satisfaction Rate (%)'].mean().reset_index()
sat_cargo.columns = ['cargo', 'satisfacao_media']

fig = px.bar(
    sat_cargo,
    x='cargo', y='satisfacao_media',
    title='Satisfação Média por Cargo',
    labels={'satisfacao_media': 'Satisfação Média (%)', 'cargo': 'Cargo'},
    category_orders={'cargo': ordem_cargos},
    color='satisfacao_media',
    color_continuous_scale=['#ef5350', '#ffa726', '#66bb6a'],
    text=sat_cargo['satisfacao_media'].round(1).astype(str) + '%'
)
fig.update_traces(textposition='outside')
fig.update_layout(coloraxis_showscale=False)
fig.add_hline(y=df['Satisfaction Rate (%)'].mean(), line_dash='dash', line_color='gray', annotation_text='Média geral')
fig.show()

## 3. Satisfação × Produtividade

In [5]:
corr, pvalor = stats.pearsonr(df['Satisfaction Rate (%)'], df['Productivity (%)'])

fig = px.scatter(
    df,
    x='Satisfaction Rate (%)', y='Productivity (%)',
    color='Department',
    title=f'Satisfação × Produtividade (correlação: {corr:.2f})',
    labels={'Satisfaction Rate (%)': 'Satisfação (%)', 'Productivity (%)': 'Produtividade (%)'},
    trendline='ols'
)
fig.show()

print(f'Correlação de Pearson: {corr:.3f}')
print(f'P-valor: {pvalor:.4f} ({"significativo" if pvalor < 0.05 else "não significativo"})')

Correlação de Pearson: 0.050
P-valor: 0.4829 (não significativo)


## 4. Satisfação × Feedback Score

In [6]:
corr_fb, pvalor_fb = stats.pearsonr(df['Satisfaction Rate (%)'], df['Feedback Score'])

# agrupar feedback em faixas
df['faixa_feedback'] = pd.cut(
    df['Feedback Score'],
    bins=[0, 2, 3.5, 5],
    labels=['Baixo (1-2)', 'Médio (2-3.5)', 'Alto (3.5-5)']
)

sat_fb = df.groupby('faixa_feedback', observed=True)['Satisfaction Rate (%)'].mean().reset_index()
sat_fb.columns = ['faixa_feedback', 'satisfacao_media']

fig = px.bar(
    sat_fb,
    x='faixa_feedback', y='satisfacao_media',
    title='Satisfação por Nível de Feedback Recebido',
    labels={'faixa_feedback': 'Feedback Score', 'satisfacao_media': 'Satisfação Média (%)'},
    color='satisfacao_media',
    color_continuous_scale=['#ef5350', '#ffa726', '#66bb6a'],
    text=sat_fb['satisfacao_media'].round(1).astype(str) + '%'
)
fig.update_traces(textposition='outside')
fig.update_layout(coloraxis_showscale=False)
fig.show()

print(f'Correlação satisfação × feedback: {corr_fb:.3f} (p={pvalor_fb:.4f})')

Correlação satisfação × feedback: 0.008 (p=0.9097)


## 5. Satisfação × Salário

In [7]:
corr_sal, pvalor_sal = stats.pearsonr(df['Satisfaction Rate (%)'], df['Salary'])

# quartis de salário
df['quartil_salario'] = pd.qcut(
    df['Salary'], q=4,
    labels=['Q1 (menor)', 'Q2', 'Q3', 'Q4 (maior)']
)

sat_sal = df.groupby('quartil_salario', observed=True)['Satisfaction Rate (%)'].mean().reset_index()
sat_sal.columns = ['quartil', 'satisfacao_media']

fig = px.bar(
    sat_sal,
    x='quartil', y='satisfacao_media',
    title='Satisfação por Faixa Salarial (Quartis)',
    labels={'quartil': 'Faixa Salarial', 'satisfacao_media': 'Satisfação Média (%)'},
    color='satisfacao_media',
    color_continuous_scale=['#ef5350', '#ffa726', '#66bb6a'],
    text=sat_sal['satisfacao_media'].round(1).astype(str) + '%'
)
fig.update_traces(textposition='outside')
fig.update_layout(coloraxis_showscale=False)
fig.show()

print(f'Correlação satisfação × salário: {corr_sal:.3f} (p={pvalor_sal:.4f})')

Correlação satisfação × salário: -0.018 (p=0.7970)


## 6. Satisfação × Tempo de Casa

In [8]:
bins = [0, 2, 5, 10, 15, 100]
labels_bins = ['< 2 anos', '2-5 anos', '5-10 anos', '10-15 anos', '> 15 anos']
df['faixa_tempo'] = pd.cut(df['anos_empresa'], bins=bins, labels=labels_bins)

sat_tempo = df.groupby('faixa_tempo', observed=True)['Satisfaction Rate (%)'].mean().reset_index()
sat_tempo.columns = ['faixa', 'satisfacao_media']

fig = px.line(
    sat_tempo,
    x='faixa', y='satisfacao_media',
    title='Satisfação por Tempo de Casa',
    labels={'faixa': 'Tempo de Casa', 'satisfacao_media': 'Satisfação Média (%)'},
    markers=True
)
fig.update_traces(line_color='#2196F3', marker_size=10)
fig.add_hline(y=df['Satisfaction Rate (%)'].mean(), line_dash='dash', line_color='gray', annotation_text='Média geral')
fig.show()

## 7. Mapa de Calor — Correlações

In [9]:
colunas_num = ['Satisfaction Rate (%)', 'Productivity (%)', 'Feedback Score', 'Salary', 'Age', 'anos_empresa', 'Projects Completed']
nomes_pt = ['Satisfação', 'Produtividade', 'Feedback', 'Salário', 'Idade', 'Tempo de Casa', 'Projetos']

corr_matrix = df[colunas_num].corr()
corr_matrix.index = nomes_pt
corr_matrix.columns = nomes_pt

fig = px.imshow(
    corr_matrix,
    title='Correlação entre Variáveis',
    color_continuous_scale='RdBu',
    zmin=-1, zmax=1,
    text_auto='.2f'
)
fig.show()

## 8. Resumo dos Achados

In [10]:
print('=== ACHADOS PRINCIPAIS ===')
print(f'Satisfação média geral: {df["Satisfaction Rate (%)"].mean():.1f}%')
print(f'Colaboradores com satisfação < 33%: {(df["Satisfaction Rate (%)"] < 33).sum()} ({(df["Satisfaction Rate (%)"] < 33).mean()*100:.1f}%)')

print(f'\nDepartamento menos satisfeito: {sat_depto.iloc[0]["departamento"]} ({sat_depto.iloc[0]["satisfacao_media"]:.1f}%)')
print(f'Departamento mais satisfeito: {sat_depto.iloc[-1]["departamento"]} ({sat_depto.iloc[-1]["satisfacao_media"]:.1f}%)')

print(f'\nCorrelação satisfação × produtividade: {corr:.3f}')
print(f'Correlação satisfação × feedback: {corr_fb:.3f}')
print(f'Correlação satisfação × salário: {corr_sal:.3f}')

=== ACHADOS PRINCIPAIS ===
Satisfação média geral: 49.9%
Colaboradores com satisfação < 33%: 63 (31.5%)

Departamento menos satisfeito: Marketing (46.0%)
Departamento mais satisfeito: IT (54.3%)

Correlação satisfação × produtividade: 0.050
Correlação satisfação × feedback: 0.008
Correlação satisfação × salário: -0.018
